# Question B2 (10 marks)
In Question B1, we used the Category Embedding model. This creates a feedforward neural network in which the categorical features get learnable embeddings. In this question, we will make use of a library called Pytorch-WideDeep. This library makes it easy to work with multimodal deep-learning problems combining images, text, and tables. We will just be utilizing the deeptabular component of this library through the TabMlp network:

In [1]:
!pip install pytorch-widedeep

INFO: pip is looking at multiple versions of opencv-contrib-python to determine which version is compatible with other requirements. This could take a while.
  Using cached click-8.3.1-py3-none-any.whl.metadata (2.6 kB)
   ---------------------------------------- 0.0/22.0 MB ? eta -:--:--
   ---- ----------------------------------- 2.4/22.0 MB 11.2 MB/s eta 0:00:02
   --------- ------------------------------ 5.0/22.0 MB 11.2 MB/s eta 0:00:02
   ----------- ---------------------------- 6.6/22.0 MB 10.1 MB/s eta 0:00:02
   ------------- -------------------------- 7.6/22.0 MB 8.9 MB/s eta 0:00:02
   ----------------- ---------------------- 9.4/22.0 MB 8.9 MB/s eta 0:00:02
   --------------------- ------------------ 11.5/22.0 MB 9.0 MB/s eta 0:00:02
   ------------------------ --------------- 13.6/22.0 MB 9.2 MB/s eta 0:00:01
   --------------------------- ------------ 15.2/22.0 MB 9.0 MB/s eta 0:00:01
   ------------------------------- -------- 17.0/22.0 MB 8.9 MB/s eta 0:00:01
   -------

  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.37.1 requires packaging<25,>=20, but you have packaging 26.0 which is incompatible.


In [3]:
SEED = 42

import os

import random
random.seed(SEED)

import numpy as np
np.random.seed(SEED)

import pandas as pd

from pytorch_widedeep.preprocessing import TabPreprocessor
from pytorch_widedeep.models import TabMlp, WideDeep
from pytorch_widedeep import Trainer
from pytorch_widedeep.metrics import R2Score

1.Divide the dataset (‘hdb_price_prediction.csv’) into train and test sets by using entries from the year 2020 and before as training data, and entries from 2021 and after as the test data.

In [5]:
df = pd.read_csv('hdb_price_prediction.csv')

# YOUR CODE HERE

# Filter Out data from year 2022 and 2023
# Define features that should be used
numeric_features     = ['dist_to_nearest_stn', 'dist_to_dhoby', 'degree_centrality', 'eigenvector_centrality', 'remaining_lease_years', 'floor_area_sqm']
categorical_features = ['month', 'town', 'flat_model_type', 'storey_range']
prediction_target    = ['resale_price']
all_features         = numeric_features + categorical_features + prediction_target

# Split features for train, validation & test by year
train_df = df[df['year'] <= 2020]
test_df  = df[df['year'] >= 2021]

# Separate features and target
train_df = train_df[all_features]
test_df  = test_df[all_features]

In [6]:
print("Train Data Shape:", train_df.shape)
print("Test Data Shape:", test_df.shape)

Train Data Shape: (87370, 11)
Test Data Shape: (72183, 11)


2.Refer to the documentation of Pytorch-WideDeep and perform the following tasks:
https://pytorch-widedeep.readthedocs.io/en/latest/index.html
* Use [**TabPreprocessor**](https://pytorch-widedeep.readthedocs.io/en/latest/examples/01_preprocessors_and_utils.html#2-tabpreprocessor) to create the deeptabular component using the continuous
features and the categorical features. Use this component to transform the training dataset.
* Create the [**TabMlp**](https://pytorch-widedeep.readthedocs.io/en/latest/pytorch-widedeep/model_components.html#pytorch_widedeep.models.tabular.mlp.tab_mlp.TabMlp) model with 2 linear layers in the MLP, with 250 and 150 neurons respectively.
* Create a [**Trainer**](https://pytorch-widedeep.readthedocs.io/en/latest/pytorch-widedeep/trainer.html#pytorch_widedeep.training.Trainer) for the training of the created TabMlp model with the mean squared error (MSE) cost function. Train the model for 80 epochs using this trainer, keeping a batch size of 64. (Note: set the *num_workers* parameter to 0.)

In [8]:
# YOUR CODE & RESULT HERE
tab_preprocessor = TabPreprocessor(
    cat_embed_cols = categorical_features,
    continuous_cols = numeric_features
)

X_tab = tab_preprocessor.fit_transform(train_df)

print(X_tab.shape)

C:\Users\chooz\anaconda3\Lib\site-packages\pytorch_widedeep\preprocessing\tab_preprocessor.py:364: UserWarning: Continuous columns will not be normalised
  warnings.warn("Continuous columns will not be normalised")


(87370, 10)


In [11]:
tab_mlp = TabMlp(
    column_idx = tab_preprocessor.column_idx,
    cat_embed_input = tab_preprocessor.cat_embed_input,
    continuous_cols = numeric_features,
    mlp_hidden_dims = [250, 150]
)

model = WideDeep(deeptabular = tab_mlp)

In [ ]:
trainer = Trainer(
    model,
    objective = 'mse',
    num_workers = 0,
    seed = SEED
)

# Train the model
trainer.fit(
    X_tab = X_tab,
    target = train_df['resale_price'].values,
    n_epochs = 80,
    batch_size = 64,
)

epoch 61:  57%|█████▋    | 778/1366 [00:14<00:11, 50.12it/s, loss=2e+9]   

3.Report the test MSE and the test R2 value that you obtained.

In [ ]:
# YOUR CODE & RESULT HERE